# Colab Setup with Google Drive

⚠️ **此 notebook 必须在 Colab 中运行，不能在 Kaggle 中运行！**

Kaggle 输入目录是只读的，会导致 `OSError: [Errno 30] Read-only file system`

**正确的使用场景**:
- 🔵 **Colab** → 此 notebook + 其他开发 notebook
- 🟠 **Kaggle** → 仅使用 `kaggle_submission.ipynb`

---

**建议方案**: 使用 Google Drive 持久化存储数据和模型

优点：
- ✅ Session 断开后数据不丢失
- ✅ 多个 notebook 间共享数据
- ✅ kaggle.json 存一次，永久使用
- ✅ 节省 Colab 临时磁盘空间（只有 ~200GB，之后自动删除）

## Step 1: 环境检查

In [ ]:
import os
from pathlib import Path

# ⚠️ 必须在 Colab 中运行
IN_KAGGLE = os.path.exists("/kaggle")
if IN_KAGGLE:
    raise RuntimeError(
        "❌ 此 notebook 必须在 Colab 中运行，不能在 Kaggle 中运行！\n"
        "Kaggle 输入目录 /kaggle/input/ 是只读的。\n"
        "\n请在 Colab (https://colab.research.google.com) 中打开此 notebook"
    )

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
if not IN_COLAB:
    print("⚠️ 警告: 未在 Colab 中检测到")
else:
    print("✓ 在 Colab 中运行")
    if os.path.exists("/content/drive"):
        print("✓ Google Drive 已挂载")
    else:
        print("ⓘ Google Drive 未挂载（将在 Step 2 挂载）")

## Step 2: 挂载 Google Drive

In [ ]:
from google.colab import drive

if not os.path.exists('/content/drive'):
    # 挂载 Google Drive（会弹出授权窗口）
    drive.mount('/content/drive')
    print("✓ Google Drive 已挂载")
else:
    print("✓ Google Drive 已挂载")

## Step 3: 准备 Google Drive 目录结构

In [ ]:
from pathlib import Path

# 创建项目目录结构
GDRIVE_BASE = Path("/content/drive/MyDrive/3drna_cc")
GDRIVE_BASE.mkdir(parents=True, exist_ok=True)

dirs = [
    GDRIVE_BASE / "data",
    GDRIVE_BASE / "models",
    GDRIVE_BASE / "output",
    GDRIVE_BASE / "credentials",
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ {d.name}")

print(f"\n项目目录: {GDRIVE_BASE}")

## Step 4: 上传 kaggle.json 到 Google Drive（一次性）

In [ ]:
from google.colab import files
import shutil

cred_dir = Path("/content/drive/MyDrive/3drna_cc/credentials")
kaggle_json = cred_dir / "kaggle.json"

if kaggle_json.exists():
    print("✓ kaggle.json 已在 Google Drive")
else:
    print("请从 https://www.kaggle.com/account/login")
    print("  → Settings → Account → Create New API Token")
    print("  下载 kaggle.json，然后点击上传:")
    uploaded = files.upload()
    
    if "kaggle.json" in uploaded:
        shutil.move("kaggle.json", str(kaggle_json))
        print(f"✓ 已保存到 Google Drive: {kaggle_json}")
    else:
        print("✗ 未找到 kaggle.json")

## Step 5: 配置 Kaggle API（每个 session 执行一次）

In [ ]:
import os
from pathlib import Path
import shutil

# 从 Google Drive 复制 kaggle.json 到 Colab 默认位置
gdrive_kaggle = Path("/content/drive/MyDrive/3drna_cc/credentials/kaggle.json")
colab_kaggle = Path("/root/.kaggle/kaggle.json")

if gdrive_kaggle.exists():
    Path("/root/.kaggle").mkdir(exist_ok=True)
    shutil.copy(str(gdrive_kaggle), str(colab_kaggle))
    os.chmod(str(colab_kaggle), 0o600)
    print("✓ kaggle.json 已配置")
    # 验证
    !kaggle competitions list | head -3
else:
    print("✗ 未找到 kaggle.json，请先上传到 Google Drive")

## Step 6: 克隆代码库到 Colab 临时存储

In [ ]:
import subprocess
import os

# 代码库放在 Colab 临时存储（快速）
REPO_DIR = "/content/3drna_cc"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/scyoyo/3drna_cc.git {REPO_DIR}
    print(f"✓ 代码库克隆到 {REPO_DIR}")
else:
    print(f"✓ 代码库已存在于 {REPO_DIR}")
    # 更新到最新
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## Step 7: 安装依赖

In [ ]:
!pip install -q -r requirements.txt
print("✓ 依赖安装完成")

# 验证关键包
import torch
print(f"✓ PyTorch {torch.__version__}")
print(f"✓ CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## Step 8: 验证配置（关键！）

In [ ]:
from src.config import IN_COLAB, GDRIVE_AVAILABLE, DATA_DIR, OUTPUT_DIR, MODEL_DIR

print(f"环境检测:")
print(f"  IN_COLAB: {IN_COLAB}")
print(f"  GDRIVE_AVAILABLE: {GDRIVE_AVAILABLE}")
print()
print(f"数据路径配置:")
print(f"  DATA_DIR: {DATA_DIR}")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")
print(f"  MODEL_DIR: {MODEL_DIR}")
print()
print(f"Google Drive 可用空间:")
!df -h /content/drive | tail -1 | awk '{print "  " $4 " 可用"}'

## Step 9: 下载竞赛数据到 Google Drive（第一次）

In [ ]:
import os
from src.config import DATA_DIR
from pathlib import Path

# 检查数据是否已存在
data_files = [
    DATA_DIR / "train_sequences.csv",
    DATA_DIR / "train_labels.csv",
    DATA_DIR / "validation_sequences.csv",
]

if all(f.exists() for f in data_files):
    print("✓ 竞赛数据已在 Google Drive")
else:
    print("下载竞赛数据到 Google Drive...")
    print("💾 首次下载 ~50GB (取决于网络，大约 1-2 小时)")
    
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    # 仅下载必要文件（不下载 310GB 的 PDB_RNA）
    !kaggle competitions download -c stanford-rna-3d-folding-2 \
      -p {DATA_DIR} \
      -f train_sequences.csv \
      -f train_labels.csv \
      -f validation_sequences.csv \
      -f validation_labels.csv \
      -f test_sequences.csv \
      -f sample_submission.csv
    
    # 解压
    for zf in DATA_DIR.glob("*.zip"):
        !unzip -q {zf} -d {DATA_DIR}
        zf.unlink()
    
    print("✓ 数据下载完成！")
    print(f"\n下载的文件:")
    !ls -lh {DATA_DIR}/*.csv 2>/dev/null | awk '{print $9, $5}'

## Step 10: 完整性检查

In [ ]:
from pathlib import Path
import pandas as pd
import os

print("="*60)
print("✓ Colab + Google Drive 设置完成！")
print("="*60)
print()

# 数据统计
from src.config import DATA_DIR
try:
    train_seq = pd.read_csv(DATA_DIR / "train_sequences.csv")
    train_labels = pd.read_csv(DATA_DIR / "train_labels.csv")
    
    print(f"数据统计:")
    print(f"  Train 序列: {len(train_seq)}")
    print(f"  Train 标签行: {len(train_labels)}")
    print()
except Exception as e:
    print(f"⚠️ 无法加载数据: {e}")

# 磁盘占用
GDRIVE_BASE = Path("/content/drive/MyDrive/3drna_cc")
if GDRIVE_BASE.exists():
    !du -sh {GDRIVE_BASE}

print()
print("="*60)
print("下一步: 打开 01_setup_and_explore.ipynb 继续!")
print("="*60)